In [ ]:
# Set your Kaggle API key as an environment variable
import os
os.environ['KAGGLE_USERNAME'] = "mabubakar35201"
os.environ['KAGGLE_KEY'] = "a0587c75f21277c97cf9253031dbea6e"

In [ ]:
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
kaggle.api.authenticate()

In [ ]:
api.dataset_download_files(
  'muhammadhananasghar/human-emotions-datasethes',
  path='/content',
  unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/muhammadhananasghar/human-emotions-datasethes


In [ ]:
os.system('rm -rf /content/EmotionsDataset')
os.system('rm -rf /content/EmotionsDataset_Splitted')
os.system('rm -rf /content/sample_data')

0

In [ ]:
train_dir = '/content/Emotions Dataset/Emotions Dataset/train'
test_dir = '/content/Emotions Dataset/Emotions Dataset/test'

In [ ]:
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class_names = ['angry', 'happy', 'sad']

In [ ]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
  train_dir,
  labels='inferred',
  label_mode='int',
  class_names=class_names,
  color_mode='rgb',
  batch_size=None,
  image_size=(256, 256),
  shuffle=True,
  seed=101,
  data_format='channels_last'
)

Found 6799 files belonging to 3 classes.


In [ ]:
val_dataset, test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
  test_dir,
  labels='inferred',
  label_mode='int',
  class_names=class_names,
  color_mode='rgb',
  image_size=(256, 256),
  batch_size=None,
  shuffle=True,
  seed=101,
  validation_split=0.5,
  subset='both',
  data_format='channels_last'
)

Found 2278 files belonging to 3 classes.
Using 1139 files for training.
Using 1139 files for validation.


In [ ]:
rescale_layer = tf.keras.layers.Rescaling(1/255)

def rescale(image, label):
    image = rescale_layer(image)
    return image, label

train_dataset = train_dataset.map(rescale)
val_dataset = val_dataset.map(rescale)
test_dataset = test_dataset.map(rescale)

In [ ]:
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(1).prefetch(tf.data.AUTOTUNE)

In [ ]:
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization
from keras.losses import SparseCategoricalCrossentropy
from keras.optimizers import Adam
from keras.metrics import SparseCategoricalAccuracy
import keras_tuner as kt

In [ ]:
class BayesianTuner(kt.HyperModel):
    def __init__(self, train_dataset, val_dataset):
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset

    def build(self, hp):
        # Define hyperparameters search space

        n_conv_layer = hp.Int('n_conv_layer', min_value=2, max_value=7, step=1)
        n_dense_layer = hp.Int('n_dense_layer', min_value=1, max_value=4, step=1)

        conv_filters = {
            1: hp.Choice('conv_filters_1', values=[4, 8, 16]),
            2: hp.Choice('conv_filters_2', values=[16, 32, 64]),
            3: hp.Choice('conv_filters_3', values=[32, 64]),
            4: hp.Choice('conv_filters_4', values=[32, 64]),
            5: hp.Choice('conv_filters_5', values=[32, 64]),
            6: hp.Choice('conv_filters_6', values=[32, 64, 128]),
            7: hp.Choice('conv_filters_7', values=[32, 64, 128])
        }

        dense_units = {
            1: hp.Choice('dense_units_1', values=[1000, 500, 100]),
            2: hp.Choice('dense_units_2', values=[500, 250, 100]),
            3: hp.Choice('dense_units_3', values=[250, 100, 50]),
            4: hp.Choice('dense_units_4', values=[100, 50, 10]),
        }

        # Create the model
        input = Input(shape=(256, 256, 3))
        x = input
        for i in range(1, n_conv_layer + 1):
            x = Conv2D(filters=conv_filters[i],
                      kernel_size=3,
                      strides=1,
                      padding='valid',
                      activation="relu",
                      use_bias=False)(x)
            x = BatchNormalization()(x)
            x = MaxPool2D(pool_size=2, strides=2)(x)

        x = Flatten()(x)

        for i in range(1, n_dense_layer + 1):
            x = Dense(
                units=dense_units[i],
                activation='relu'
            )(x)
            x = BatchNormalization()(x)
        x = Dense(3, activation='softmax')(x)

        model = tf.keras.Model(inputs=input, outputs=x)

        # Compile model
        model.compile(
            optimizer=Adam(),
            loss=SparseCategoricalCrossentropy(),
            metrics=[SparseCategoricalAccuracy()]
        )

        return model

    def fit(self, hp, model, *args, **kwargs):

        return model.fit(train_dataset, validation_data = val_dataset, **kwargs)

In [ ]:
# Initialize the tuner
tuner = kt.BayesianOptimization(
    hypermodel=BayesianTuner(train_dataset, val_dataset),
    objective='val_sparse_categorical_accuracy',
    max_trials=15,
    directory='bayesian_opt',
    project_name='human_emotions_detection'
)

In [ ]:
learning_rate_callback = tf.keras.callbacks.ReduceLROnPlateau(
  monitor='val_sparse_categorical_accuracy',
  factor=0.3,
  patience=3,
  min_lr=1e-6,
  min_delta=0.01,
)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_sparse_categorical_accuracy',           # Metric to monitor
            min_delta=0.01,             # Minimum change to qualify as an improvement
            patience=5,                   # Number of epochs with no improvement after which training will stop
            verbose=1,                    # Print messages about early stopping
            mode='max',                  # The direction is max because we're monitoring accuracy
            restore_best_weights=True,   # Restore model weights from the epoch with the best value of the monitored quantity
        )

In [ ]:
# Search for best hyperparameters
tuner.search(
    epochs=15,
    callbacks=[
        early_stopping,
        learning_rate_callback
    ]
)

In [ ]:
# Get best hyperparameters and model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Print best hyperparameters
print("Best hyperparameters found:")
print(f"number of convolution layers: {best_hps.get('n_conv_layer')}")
print(f"number of dense layers: {best_hps.get('n_dense_layer')}")
print(f"Conv Filters 1: {best_hps.get('conv_filters_1')}")
print(f"Conv Filters 2: {best_hps.get('conv_filters_2')}")
print(f"Conv Filters 3: {best_hps.get('conv_filters_3')}")
print(f"Conv Filters 4: {best_hps.get('conv_filters_4')}")
print(f"Conv Filters 5: {best_hps.get('conv_filters_5')}")
print(f"Conv Filters 6: {best_hps.get('conv_filters_6')}")
print(f"Conv Filters 7: {best_hps.get('conv_filters_7')}")
print(f"Dense Units 1: {best_hps.get('dense_units_1')}")
print(f"Dense Units 2: {best_hps.get('dense_units_2')}")
print(f"Dense Units 3: {best_hps.get('dense_units_3')}")
print(f"Dense Units 4: {best_hps.get('dense_units_4')}")

Hyperparameter Tuning Results:

Conv Filters 1: 8

Conv Filters 2: 16

Conv Filters 3: 64

Conv Filters 4: 32

Conv Filters 5: 32

Conv Filters 7: 128

Dense Units 1: 1000

Dense Units 2: 500

Dense Units 3: 50

Dense Units 4: 10